In [1]:
import sqlite3
import pandas as pd

# Load the historical attempts ordered by time
conn = sqlite3.connect('dsa_database.db')
query = "SELECT candidate_id, problem_id, passed FROM candidate_attempts ORDER BY timestamp ASC"
df_attempts = pd.read_sql_query(query, conn)
conn.close()

# Initialize base Elo ratings
candidate_elo = {cid: 1200.0 for cid in df_attempts['candidate_id'].unique()}
problem_elo = {pid: 1200.0 for pid in df_attempts['problem_id'].unique()}

K = 32 # The maximum rating change per attempt

# Process each attempt chronologically
for index, row in df_attempts.iterrows():
    cid = row['candidate_id']
    pid = row['problem_id']
    actual_score = row['passed'] # 1 for pass, 0 for fail
    
    R_cand = candidate_elo[cid]
    R_prob = problem_elo[pid]
    
    # Calculate expected probabilities
    E_cand = 1 / (1 + 10 ** ((R_prob - R_cand) / 400))
    E_prob = 1 - E_cand # The probability that the problem defeats the candidate
    
    # Update ratings
    candidate_elo[cid] = round(R_cand + K * (actual_score - E_cand), 2)
    
    # The problem's actual score is the inverse of the candidate's score
    problem_actual = 1 - actual_score
    problem_elo[pid] = round(R_prob + K * (problem_actual - E_prob), 2)

# Convert results to DataFrames for analysis
df_cand_elo = pd.DataFrame(list(candidate_elo.items()), columns=['candidate_id', 'elo_rating'])
df_cand_elo = df_cand_elo.sort_values(by='elo_rating', ascending=False).reset_index(drop=True)

df_prob_elo = pd.DataFrame(list(problem_elo.items()), columns=['problem_id', 'elo_rating'])
df_prob_elo = df_prob_elo.sort_values(by='elo_rating', ascending=False).reset_index(drop=True)

print("Elo Baseline Complete.")
print("\nTop 5 Strongest Candidates:")
display(df_cand_elo.head())

print("\nTop 5 Most Difficult Problems (Highest Rating):")
display(df_prob_elo.head())

Elo Baseline Complete.

Top 5 Strongest Candidates:


,candidate_id,elo_rating
0,cand_137,1686.79
1,cand_192,1624.58
2,cand_086,1606.83
3,cand_125,1602.25
4,cand_034,1592.93



Top 5 Most Difficult Problems (Highest Rating):


,problem_id,elo_rating
0,1398,1447.86
1,3514,1446.66
2,2905,1445.80
3,685,1443.26
4,3548,1440.76


In [2]:
import sqlite3
import pandas as pd
import numpy as np

# Load attempts with topic information
conn = sqlite3.connect('dsa_database.db')
query = """
SELECT candidate_id, problem_id, topic, passed 
FROM candidate_attempts 
ORDER BY timestamp ASC
"""
df_attempts = pd.read_sql_query(query, conn)

# Identify top primary topics
top_topics = df_attempts['topic'].value_counts().head(6).index.tolist()
df_filtered = df_attempts[df_attempts['topic'].isin(top_topics)].copy()

# Initialize topic-wise skill parameters (\theta) for each candidate
candidates = df_filtered['candidate_id'].unique()
topic_skills = {cid: {topic: 0.0 for topic in top_topics} for cid in candidates}
item_difficulty = {pid: 0.0 for pid in df_filtered['problem_id'].unique()}

learning_rate = 0.1

# Iterate to estimate ability (\theta) per topic using 1PL logit gradient updates
for _, row in df_filtered.iterrows():
    cid = row['candidate_id']
    pid = row['problem_id']
    topic = row['topic']
    y = row['passed']  # 1 or 0
    
    theta = topic_skills[cid][topic]
    b = item_difficulty[pid]
    
    # Sigmoid / Logistic function
    p_hat = 1 / (1 + np.exp(-(theta - b)))
    
    # Gradient step for 1PL IRT estimation
    error = y - p_hat
    topic_skills[cid][topic] += learning_rate * error
    item_difficulty[pid] -= learning_rate * error

# Convert topic skills into a structured DataFrame (Candidate Skill Matrix)
skill_matrix = pd.DataFrame.from_dict(topic_skills, orient='index')
skill_matrix.index.name = 'candidate_id'
skill_matrix = skill_matrix.round(3)

# Merge with overall Elo rating for comparison
skill_matrix = skill_matrix.merge(df_cand_elo, on='candidate_id')

print("Module 3 IRT Complete: Candidate Topic Skill Matrix Generated.")
print("\nSample Skill Profiles (First 5 Candidates):")
display(skill_matrix.head())

# Save skill matrix to SQLite database for Module 4 & 5 consumption
skill_matrix.to_sql('candidate_skill_profiles', conn, if_exists='replace', index=False)
conn.close()

Module 3 IRT Complete: Candidate Topic Skill Matrix Generated.

Sample Skill Profiles (First 5 Candidates):


,candidate_id,Array,String,Math,Hash Table,Database,Tree,elo_rating
0,cand_002,0.415,-0.185,-0.102,-0.007,0.429,-0.013,1355.15
1,cand_200,1.108,0.273,0.028,0.361,-0.018,0.183,1432.32
2,cand_083,0.488,0.189,0.261,0.020,0.056,-0.057,1502.51
3,cand_014,-0.186,-0.131,0.037,-0.019,0.042,0.150,1135.97
4,cand_142,-0.032,0.173,0.068,0.115,0.293,0.111,927.76
